In [2]:
import sys
from pathlib import Path
here = Path.cwd()
ROOT = next(p for p in [here, *here.parents] if (p / "src" / "data.py").exists())
sys.path.insert(0, str(ROOT))

from src.data import load_raw, build_orders, ORDERS_PATH

raw = load_raw()
orders = build_orders(raw)

shipped = raw[raw["Delivery Status"] != "Shipping canceled"]
per_order = shipped.groupby("Order Id")[["Shipping Mode", "Days for shipment (scheduled)", "Market", "Customer Segment", "Type"]].nunique()
real_over = shipped["Days for shipping (real)"] > shipped["Days for shipment (scheduled)"]

print("raw rows:", len(raw), "| canceled/fraud rows removed:", len(raw) - len(shipped))
print("orders:", len(orders), "| late rate:", round(orders["late"].mean(), 4))
print("order-level columns constant within each order:", bool((per_order == 1).all().all()))
print("late flag == (real > scheduled) on shipped rows:", round((real_over == shipped["Late_delivery_risk"]).mean(), 4))
orders.head()

raw rows: 180519 | canceled/fraud rows removed: 7754
orders: 62897 | late rate: 0.5731
order-level columns constant within each order: True
late flag == (real > scheduled) on shipped rows: 1.0


,Order Id,order_date,shipping_mode,scheduled_days,market,region,country,segment,payment_type,main_department,n_items,n_departments,total_qty,order_value,avg_discount_rate,late,order_month,order_dow,order_hour
0,1,2015-01-01 00:00:00,Standard Class,4,LATAM,Central America,México,Consumer,CASH,Fan Shop,1,1,1,239.979996,0.200000,0,1,3,0
1,2,2015-01-01 00:21:00,Standard Class,4,LATAM,South America,Colombia,Consumer,PAYMENT,Fan Shop,3,3,7,529.380005,0.096667,0,1,3,0
2,4,2015-01-01 01:03:00,Standard Class,4,LATAM,South America,Colombia,Home Office,CASH,Outdoors,4,4,14,620.870014,0.142500,1,1,3,1
3,5,2015-01-01 01:24:00,Standard Class,4,LATAM,South America,Colombia,Consumer,DEBIT,Apparel,5,2,10,987.070007,0.136000,1,1,3,1
4,7,2015-01-01 02:06:00,Second Class,2,LATAM,South America,Brasil,Consumer,DEBIT,Fan Shop,3,2,7,525.520004,0.070000,1,1,3,2


In [3]:
orders.to_csv(ORDERS_PATH, index=False)
print("saved", ORDERS_PATH.name, orders.shape)

saved orders.csv (62897, 19)
